In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain.document_loaders import PyPDFLoader

In [3]:
loader_harrypotter  = PyPDFLoader("../harrypotter_1.pdf")
documnet_harrypotter = loader_harrypotter.load()

In [4]:
print(len(documnet_harrypotter))

250


In [5]:
loader_rag = PyPDFLoader("../Retrieval-Augmented-Generation.pdf")
documnet_rag = loader_rag.load()

In [6]:
print(len(documnet_rag))

19


In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [9]:
text_harrypotter = text_splitter.split_documents(documnet_harrypotter)
text_harrypotter = [doc.page_content for doc in text_harrypotter]

In [10]:
text_rag = text_splitter.split_documents(documnet_rag)
text_rag = [doc.page_content for doc in text_rag]

In [11]:
print(len(text_harrypotter))

1155


In [12]:
print(len(text_rag))

183


In [13]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [14]:
from langchain.vectorstores import Chroma
import chromadb

In [15]:
DB_DIR = '../content/db'
DB_DIR

'../content/db'

In [16]:
client_settings = chromadb.config.Settings(
    is_persistent=True,
    persist_directory=DB_DIR,
    anonymized_telemetry=False,
)

In [18]:
harrypotter_vectorstore = Chroma.from_texts(
    text_harrypotter,
    embeddings,
    client_settings=client_settings,
    collection_name="harrypotter",
    collection_metadata={"hnsw": "cosine"},
    persist_directory="./store/harrypotter",
)

In [19]:
rag_vectorstore = Chroma.from_texts(
    text_rag,
    embeddings,
    client_settings=client_settings,
    collection_name="rag",
    collection_metadata={"hnsw": "cosine"},
    persist_directory="./store/rag",
)

In [20]:
retriever_harrypotter = harrypotter_vectorstore.as_retriever(
    search_type="mmr", search_kwargs={"k": 5, "include_metadata": True}
)

In [21]:
retriever_rag = rag_vectorstore.as_retriever(
    search_type="mmr", search_kwargs={"k": 5, "include_metadata": True}
)

In [22]:
from langchain.retrievers.merger_retriever import MergerRetriever

In [23]:
lotr = MergerRetriever(retrievers=[retriever_harrypotter, retriever_rag])
lotr

MergerRetriever(retrievers=[VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000234F0D6EEA0>, search_type='mmr', search_kwargs={'k': 5, 'include_metadata': True}), VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000234F1229B20>, search_type='mmr', search_kwargs={'k': 5, 'include_metadata': True})])

In [25]:
for chunks in lotr.invoke("Who was the jon snow?"):
    print(chunks.page_content)
    print("\n")

212
that night you won Norbert? What did the stranger you were playing cards
with look like?"
"Dunno," said Hagrid casually, "he wouldn' take his cloak off."
He saw the three of them look stunned and raised his eyebrows.
"It's not that unusual, yeh get a lot o' funny folk in the Hog's Head --
that's the pub down in the village. Mighta bin a dragon dealer, mightn'
he? I never saw his face, he kept his hood up."
Harry sank down next to the bowl of peas. "What did you talk to him


[66] Thomas Wolf, Lysandre Debut, Victor Sanh, Julien Chaumond, Clement Delangue, Anthony
Moi, Pierric Cistac, Tim Rault, Rémi Louf, Morgan Funtowicz, Joe Davison, Sam Shleifer,
Patrick von Platen, Clara Ma, Yacine Jernite, Julien Plu, Canwen Xu, Teven Le Scao, Sylvain
Gugger, Mariama Drame, Quentin Lhoest, and Alexander M. Rush. Huggingface’s transformers:
State-of-the-art natural language processing. ArXiv, abs/1910.03771, 2019.


though his scar were on fire. Half blinded, he staggered backward. He
heard hoo

In [26]:
for chunks in lotr.invoke("Who is harry potter?"):
    print(chunks.page_content)
    print("\n")

he is?"
"Who?"
"Harry Potter!"
Harry heard the little girl's voice.
"Oh, Mom, can I go on the train and see him, Mom, eh please...."
"You've already seen him, Ginny, and the poor boy isn't something you
goggle at in a zoo. Is he really, Fred? How do you know?"
"Asked him. Saw his scar. It's really there - like lightning."
"Poor dear - no wonder he was alone, I wondered. He was ever so polite
when he asked how to get onto the platform."


The	Divine
Comedy	(x) q 
Query 
Encoder 
q(x) 
MIPS p θ 
Generator pθ
(Parametric) 
Margin- 
alize 
This	14th	century	work
is	divided	into	3
sections:	"Inferno",
"Purgatorio"	&
"Paradiso"									(y)
End-to-End Backprop through q  and p θ 
Barack	Obama	was
born	in	Hawaii.(x)
Fact Veriﬁcation: Fact Query
supports	(y)
Question Generation
Fact Veriﬁcation:
Label Generation
Document 
Index 
Define	"middle	ear"(x)
Question Answering:
Question Query
The	middle	ear	includes
the	tympanic	cavity	and


Potter.
"Look," said Harry, throwing caution to the winds, "

In [27]:
from langchain.document_transformers import EmbeddingsRedundantFilter
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain.retrievers import ContextualCompressionRetriever
from langchain.document_transformers import LongContextReorder

In [28]:
filter = EmbeddingsRedundantFilter(embeddings=embeddings)

reordering = LongContextReorder()

pipeline = DocumentCompressorPipeline(transformers=[filter, reordering])

compression_retriever_reordered = ContextualCompressionRetriever(
    base_compressor=pipeline, base_retriever=lotr, search_kwargs={"k": 3, "include_metadata": True}
)

In [29]:
compression_retriever_reordered

ContextualCompressionRetriever(base_compressor=DocumentCompressorPipeline(transformers=[EmbeddingsRedundantFilter(embeddings=OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x00000234EC376C30>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x00000234EE555760>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True), similarity_fn=<function cosine_similari

In [30]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.0
)

In [31]:
from langchain.chains import RetrievalQA

In [32]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=compression_retriever_reordered,
    return_source_documents=True,
)

In [34]:
query ="who is harry potter ?"

results = qa.invoke(query)
results

{'query': 'who is harry potter ?',
 'result': "I don't know.",
 'source_documents': [_DocumentWithState(metadata={}, page_content='The\tDivine\nComedy\t(x) q \nQuery \nEncoder \nq(x) \nMIPS p θ \nGenerator\xa0pθ\n(Parametric) \nMargin- \nalize \nThis\t14th\tcentury\twork\nis\tdivided\tinto\t3\nsections:\t"Inferno",\n"Purgatorio"\t&\n"Paradiso"\t\t\t\t\t\t\t\t\t(y)\nEnd-to-End Backprop through q  and\xa0p θ \nBarack\tObama\twas\nborn\tin\tHawaii.(x)\nFact Veriﬁcation: Fact Query\nsupports\t(y)\nQuestion Generation\nFact Veriﬁcation:\nLabel Generation\nDocument \nIndex \nDefine\t"middle\tear"(x)\nQuestion Answering:\nQuestion Query\nThe\tmiddle\tear\tincludes\nthe\ttympanic\tcavity\tand', state={'embedded_doc': [0.03430734574794769, -0.0032059960067272186, 0.02471749670803547, 0.020260242745280266, -0.02488258108496666, 0.004183365032076836, -0.04952503740787506, 0.03799921274185181, -0.024777527898550034, -0.031365856528282166, 0.06519296020269394, -0.024942610412836075, -0.083562247455

In [35]:
results['result']

"I don't know."

In [36]:
query = "What is Abstractive Question Answering?"

response = qa.invoke(query)

In [37]:
response

{'query': 'What is Abstractive Question Answering?',
 'result': "I don't know.",
 'source_documents': [_DocumentWithState(metadata={}, page_content='Open-domain question answering (QA) is an important real-world application and common testbed\nfor knowledge-intensive tasks [20]. We treat questions and answers as input-output text pairs (x,y)\nand train RAG by directly minimizing the negative log-likelihood of answers. We compare RAG to\nthe popular extractive QA paradigm [5, 7, 31, 26], where answers are extracted spans from retrieved\ndocuments, relying primarily on non-parametric knowledge. We also compare to “Closed-Book', state={'embedded_doc': [-0.008860870264470577, 0.032241810113191605, 0.015595631673932076, 0.015483072958886623, 0.012919235974550247, -0.03139136731624603, -0.03199167922139168, 0.011024498380720615, -0.006809800397604704, 0.031316328793764114, 0.02194894477725029, -0.028739985078573227, 0.002346535911783576, 0.016120905056595802, 0.02397499978542328, -0.02263680

In [38]:
for source in response["source_documents"]:
    print(source.metadata)

{}
{}
{}
{}
{}
{}
{}
{}
{}
{}
